# 可视化基础与周报

**学习目标：** 学习 Plotly 绘图（参数配置、图表大小调整）。

**实战任务：** 绘制门店评论数量分布、评分分布等数据分布图。

## 1. 环境准备

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 加载清洗后数据
df = pd.read_csv('clean_data.csv')
df['comment_time'] = pd.to_datetime(df['comment_time'])
print(f'数据形状: {df.shape}')
df.head(3)

数据形状: (26825, 15)


,cus_id,comment_time,comment_star,cus_comment,kouwei,huanjing,fuwu,shopID,stars,year,month,weekday,hour,comment_len,rating_level
0,迷糊泰迪,2018-09-20 06:48:00,sml-str40,南信 算是 广州 著名 甜品店 吧 好几个 时间段 路过 都 是 座无虚席 看着 餐单 上 ...,非常好,好,好,518986,4.0,2018,9,3,6,184,中评
1,稱霸幼稚園,2018-09-22 21:49:00,sml-str40,中午 吃 完 了 所谓 的 早茶 回去 放下 行李 休息 了 会 就 来 吃 下午茶 了 服...,很好,很好,很好,518986,4.0,2018,9,5,21,266,中评
2,爱吃的美美侠,2018-09-22 22:16:00,sml-str40,冲刺 王者 战队 吃遍 蓉城 战队 有 特权 五月份 和 好 朋友 毕业 旅行 来 了 广州...,很好,很好,很好,518986,4.0,2018,9,5,22,341,中评


## 2. Plotly 基础 — 参数与大小调整

Plotly 常用参数：
- `width` / `height`：图表尺寸（像素）
- `title`：标题
- `labels`：轴标签映射
- `color_discrete_sequence`：颜色序列
- `template`：主题模板（plotly, plotly_white, ggplot2 等）

In [2]:
# 基础直方图示例
fig = px.histogram(
    df, x='stars',
    title='评分分布（基础版）',
    nbins=10,
    width=800, height=500
)
fig.show()

## 3. 门店评论数量分布

In [3]:
# 各门店评论数量
shop_counts = df.groupby('shopID').size().reset_index(name='评论数')
shop_counts = shop_counts.sort_values('评论数', ascending=True)

fig = px.bar(
    shop_counts,
    x='评论数', y='shopID',
    orientation='h',
    title='各门店评论数量分布',
    labels={'shopID': '门店ID', '评论数': '评论数量'},
    color='评论数',
    color_continuous_scale='Blues',
    width=900, height=500
)
fig.update_layout(
    template='plotly_white',
    font=dict(size=14),
    coloraxis_showscale=False
)
fig.show()

In [4]:
# 门店评论占比饼图
fig = px.pie(
    shop_counts,
    values='评论数', names='shopID',
    title='门店评论数量占比',
    width=700, height=600,
    color_discrete_sequence=px.colors.qualitative.Set3
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

## 4. 评分分布

In [5]:
# 总体评分分布直方图
fig = px.histogram(
    df, x='stars',
    nbins=20,
    title='用户评分分布',
    labels={'stars': '评分', 'count': '评论数'},
    color_discrete_sequence=['#FF6B6B'],
    width=800, height=500
)
fig.update_layout(
    template='plotly_white',
    bargap=0.1,
    xaxis=dict(tickmode='linear', tick0=1, dtick=0.5)
)
fig.show()

In [6]:
# 评分等级分布
rating_dist = df['rating_level'].value_counts().reset_index()
rating_dist.columns = ['评分等级', '数量']

fig = px.bar(
    rating_dist, x='评分等级', y='数量',
    title='评分等级分布（好评/中评/差评）',
    color='评分等级',
    color_discrete_map={'好评': '#2ECC71', '中评': '#F39C12', '差评': '#E74C3C'},
    width=700, height=450
)
fig.update_layout(showlegend=False, template='plotly_white')
fig.show()

## 5. 多维度评分分布（口味/环境/服务）

In [7]:
dim_cols = ['kouwei', 'huanjing', 'fuwu']
dim_data = []
for col in dim_cols:
    counts = df[col].value_counts().reset_index()
    counts.columns = ['评分', '数量']
    counts['维度'] = {'kouwei': '口味', 'huanjing': '环境', 'fuwu': '服务'}[col]
    dim_data.append(counts)
dim_df = pd.concat(dim_data, ignore_index=True)

fig = px.bar(
    dim_df, x='评分', y='数量', color='维度',
    barmode='group',
    title='口味 / 环境 / 服务 评分分布对比',
    width=900, height=500,
    color_discrete_sequence=['#3498DB', '#9B59B6', '#1ABC9C']
)
fig.update_layout(template='plotly_white', legend_title='维度')
fig.show()

## 6. 组合仪表盘

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('评分分布', '评论长度分布', '各门店评论数', '评分等级'),
    specs=[[{'type': 'histogram'}, {'type': 'histogram'}],
           [{'type': 'bar'}, {'type': 'pie'}]]
)

fig.add_trace(go.Histogram(x=df['stars'], name='评分', marker_color='#FF6B6B'), row=1, col=1)
fig.add_trace(go.Histogram(x=df['comment_len'], name='评论长度', marker_color='#4ECDC4'), row=1, col=2)
fig.add_trace(go.Bar(x=shop_counts['shopID'], y=shop_counts['评论数'], marker_color='#45B7D1'), row=2, col=1)
fig.add_trace(go.Pie(labels=rating_dist['评分等级'], values=rating_dist['数量']), row=2, col=2)

fig.update_layout(
    height=800, width=1100,
    title_text='大众点评数据分布仪表盘',
    showlegend=False,
    template='plotly_white'
)
fig.show()